# 02 — Train a DQN Model Offline

Load a stored transition dataset and train a MOUSE model entirely from replayed data, with no live environment in the training loop.

A MOUSE model is assembled from three independent pieces:

- `StepEmbedder` converts each `(action, observation, reward, done)` step into a token sequence the backbone can process.
- `Qwen3Backbone` reads those tokens with a transformer and builds up context over the sequence.
- `DiscreteActionValueHead` maps the backbone's output to one Q-value per action.

`Qwen3Backbone` downloads the full `Qwen/Qwen3-0.6B` backbone from the Hub on first run. Its `hidden_dim` (1024 for this checkpoint) is the single number that connects all three pieces — no manual dimension matching required. For quick debugging you can pass `num_layers=...` to truncate the backbone, but this tuned training example uses all pretrained layers.

In [1]:
import torch

from mouse_core.data import (
    DataLoader,
    Augmenter,
    load_stores_from_hub,
)
from mouse_core.objectives import DqnObjective
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import StepEmbedder
from mouse_core.models.heads import DiscreteActionValueHead

DATASET_ID = "mouse-example-dataset"
MODEL_ID = "mouse-example-model"
FORCE_DOWNLOAD = True   # set False to use the local Hub cache
MAX_ACTIONS = 4
MAX_OBS_DISCRETE = 64
TRAIN_STEPS = 20000
SEQUENCE_LENGTH = 512
BATCH_SIZE = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load data

`load_stores_from_hub` downloads the matching dataset parquet shards as a local Hub snapshot, then reconstructs the `Datastore` objects — one per stored maze stream.

In [2]:
stores = load_stores_from_hub(DATASET_ID, split="train", force_download=FORCE_DOWNLOAD)
for s in stores:
    print(s)

Fetching ... files: 0it [00:00, ?it/s]

Datastore(name='proc_frozenlake_0', steps=1500, columns=['action', 'time', 'reward', 'done', 'episode_index', 'task_index', 'observation', 'info_prob', 'info_q_star'])
Datastore(name='proc_frozenlake_1', steps=1500, columns=['action', 'time', 'reward', 'done', 'episode_index', 'task_index', 'observation', 'info_prob', 'info_q_star'])
Datastore(name='proc_frozenlake_10', steps=1500, columns=['action', 'time', 'reward', 'done', 'episode_index', 'task_index', 'observation', 'info_prob', 'info_q_star'])
Datastore(name='proc_frozenlake_100', steps=1500, columns=['action', 'time', 'reward', 'done', 'episode_index', 'task_index', 'observation', 'info_prob', 'info_q_star'])
Datastore(name='proc_frozenlake_101', steps=1500, columns=['action', 'time', 'reward', 'done', 'episode_index', 'task_index', 'observation', 'info_prob', 'info_q_star'])
Datastore(name='proc_frozenlake_102', steps=1500, columns=['action', 'time', 'reward', 'done', 'episode_index', 'task_index', 'observation', 'info_prob', '

## DataLoader and augmentation

`DataLoader` slices one or more `Datastore`s into fixed-length sequence batches and mixes them automatically — no manual merging required.

- **`sequence_length=128`** — each item in a batch is 128 consecutive steps from a single env instance.
- **`batch_size=4`** — 4 sequences per call to `next_batch()`.
- **`prefetch=4`** — pre-loads 4 batches in the background to keep the GPU fed.

`next_batch()` returns a `list[list[dict]]` of shape `[batch][sequence]` — one inner list per sequence, one dict per step. Passing `augmenter=augmenter` makes `DataLoader` run `Augmenter` inside the sampling path. Augmentation modality specs use `field` plus an augmentation `type` such as `discrete`, `linear`, or `image`. The tuned baseline defines the augmenter but leaves it disabled in the loader.

In [3]:
augmenter = Augmenter(
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
    keep_fields=["action", "observation", "reward", "done"],
    seed=0,
)

loader = DataLoader(
    stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    prefetch=4,
    num_workers=0,
    pack=True,
    augmenter=augmenter,
)

## Build the model

All three components are sized by `backbone.hidden_dim`, so they connect without any manual dimension matching.

**`StepEmbedder`** — each entry in `modalities` describes one field from the step dict and which `type` to use:
- `"discrete"` — looks up a learned vector from a small vocabulary. Use for integer-valued fields like actions, observations, and done codes.
- `"rff"` — embeds a continuous scalar using random Fourier features. Use for rewards or any float value.
- `"learnable"` — a free learnable token with no input, giving the model scratch space it can write to without consuming an observation slot.

`include_type_token=False` disables the per-modality type embedding that would otherwise be added on top of every content embedding. With all modalities at the same `std=0.02` the type signal is already balanced, but disabling it keeps the summed token purely content-driven.

**`DiscreteActionValueHead`** — a small MLP that predicts one Q-value per action. The tuned baseline uses `scale=1.0` for the action-value head.

**`Model`** — wraps encoder, backbone, and head(s) into a single object with a unified `forward` call.

In [4]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B")

encoder = StepEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "tokens": 1
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "tokens": 1
        },
        {
            "field": "reward",
            "type": "rff",
            "std": 0.02,
            "in_min": 0.01,
            "in_max": 100.0,
            "tokens": 1
        },
        {
            "field": "done",
            "type": "discrete",
            "vocab_size": 5,
            "std": 0.02,
            "tokens": 1
        },
        {
            "type": "learnable",
            "std": 0.02,
            "tokens": 1
        },
    ],
    concat_modalities=False,
    include_type_token=False,
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device)
print(model)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Model(
  (encoder): StepEmbedder(
    (type_embedder): TypeEmbedder(
      (embed): ScaledEmbedding(64, 1024)
    )
    (modality_embedders): ModuleDict(
      (action): DiscreteEmbedder(
        (embed): ScaledEmbedding(4, 1024)
      )
      (observation): DiscreteEmbedder(
        (embed): ScaledEmbedding(64, 1024)
      )
      (reward): ScalarRFFEmbedder(
        (rff): RandomFourierFeatures()
      )
      (done): DiscreteEmbedder(
        (embed): ScaledEmbedding(5, 1024)
      )
      (__learnable_4): LearnableEmbedder()
    )
  )
  (backbone): Qwen3Backbone(
    (model): Qwen3Model(
      (embed_tokens): Embedding(1, 1024)
      (layers): ModuleList(
        (0-27): 28 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (o_proj

## Train

Each iteration:
1. Sample an already-augmented batch of transition sequences from the data loader.
2. Run both the online network and a frozen target network to get Q-value predictions.
3. Compute the Bellman loss — `objective(objective_data, predictions)` returns `(loss, metrics)`.
4. Back-propagate, clip gradients, and step the optimizer.
5. Slowly copy online weights → target network (Polyak/EMA update) for stable TD targets.

### Discount factors and done codes

MOUSE uses five `done` codes. `DqnObjective` maps each to its own bootstrap discount:

- `0`: running, using `gamma_step`.
- `1`: episode done within a task, using `gamma_episode_terminal`.
- `2`: episode truncated within a task, using `gamma_episode_truncated`.
- `3`: task done, using `gamma_task_terminal`.
- `4`: task truncated, using `gamma_task_truncated`.

The example dataset is collected with `episodes_per_task=0`, so all episodes belong to one ongoing task. We bootstrap through every done code with `0.99`, which keeps the target connected across episode and task markers instead of cutting it off.

In [5]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1.0e-5, weight_decay=0.0, betas=(0.9, 0.95), eps=1.0e-8)
objective = DqnObjective(
    gamma_step=1.0,
    gamma_episode_terminal=0.99,   # bootstrap through episode boundaries
    gamma_episode_truncated=0.99,
    gamma_task_terminal=0.99,      # bootstrap through task boundaries too
    gamma_task_truncated=0.99,
    tau=0.0005,
)

for step in range(TRAIN_STEPS):

    batch = loader.next_batch()

    predictions, objective_data, _ = model(batch)
    loss, _ = objective(objective_data, predictions)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    model.polyak_update(action_value_tau=objective.tau)

    if step % 100 == 0:
        print(f"step={step}  loss={loss}")

loader.close()
print("Training finished.")

step=0  loss=0.07658722251653671
step=100  loss=0.07248091697692871
step=200  loss=0.02479461021721363
step=300  loss=0.016920369118452072
step=400  loss=0.015915140509605408
step=500  loss=0.018085729330778122
step=600  loss=0.01218888908624649
step=700  loss=0.01024174690246582
step=800  loss=0.010834972374141216
step=900  loss=0.01093982346355915
step=1000  loss=0.009289944544434547
step=1100  loss=0.01228044368326664
step=1200  loss=0.008028818294405937
step=1300  loss=0.009993876330554485
step=1400  loss=0.01039816252887249
step=1500  loss=0.008085744455456734
step=1600  loss=0.007551509886980057
step=1700  loss=0.006180233787745237
step=1800  loss=0.007537707220762968
step=1900  loss=0.006791045889258385
step=2000  loss=0.0062384577468037605
step=2100  loss=0.006273791193962097
step=2200  loss=0.004183934535831213
step=2300  loss=0.005943169817328453
step=2400  loss=0.006111537106335163
step=2500  loss=0.005516867619007826
step=2600  loss=0.006661190185695887
step=2700  loss=0.00

## Push to the Hub

Save the offline-trained model to the shared Hub repo under `MODEL_ID`. The inference notebook loads the current checkpoint from that repo, so whichever training notebook you push last is the model it evaluates.

In [6]:
model.eval().to("cpu")
url = push_model_to_hub(model, MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")

Pushed to https://huggingface.co/micahr234/mouse-example-model
